# Malimg malware-family classification

This notebook is a cleaned, reproducible version of university coursework by Jose E. Rodriguez Rios. It removes the original Google Drive paths, account metadata, saved model binaries, and embedded outputs.

The workflow consumes **already-rendered PNG images only**. Do not execute or open malware samples on a normal workstation.

The Malimg method and dataset are described in [Nataraj et al. (2011)](https://vision.ece.ucsb.edu/sites/default/files/publications/nataraj_vizsec_2011_paper.pdf). Supply your own lawful copy of the image dataset.

## Environment and configuration

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

SEED = 123
BATCH_SIZE = 32
IMAGE_SIZE = (180, 180)
EPOCHS = int(os.environ.get("EPOCHS", "15"))

tf.keras.utils.set_random_seed(SEED)


In [ ]:
DATA_DIR = Path(os.environ.get("MALWARE_DATASET_DIR", "data/malimg"))
if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"Dataset directory not found: {DATA_DIR}. "
        "Set MALWARE_DATASET_DIR to a directory containing one subdirectory per family."
    )


## Dataset loading

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)
class_names = train_ds.class_names
print(f"Classes ({len(class_names)}): {class_names}")

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)


## Convolutional neural network

In [ ]:
def build_model(num_classes):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(*IMAGE_SIZE, 3)),
        tf.keras.layers.Rescaling(1.0 / 255),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Dense(num_classes),
    ])


model = build_model(len(class_names))
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
model.summary()


## Training

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
)


In [ ]:
epochs = range(1, len(history.history["accuracy"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history.history["accuracy"], label="training")
axes[0].plot(epochs, history.history["val_accuracy"], label="validation")
axes[0].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy")
axes[0].legend()

axes[1].plot(epochs, history.history["loss"], label="training")
axes[1].plot(epochs, history.history["val_loss"], label="validation")
axes[1].set(title="Loss", xlabel="Epoch", ylabel="Loss")
axes[1].legend()

fig.tight_layout()


## Evaluation

In [ ]:
loss, accuracy = model.evaluate(val_ds, verbose=0)
print(f"Validation loss: {loss:.4f}")
print(f"Validation accuracy: {accuracy:.4f}")

y_true = np.concatenate([labels.numpy() for _, labels in val_ds])
logits = model.predict(val_ds, verbose=0)
y_pred = np.argmax(logits, axis=1)

print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    display_labels=class_names,
    xticks_rotation=90,
    normalize="true",
    values_format=".2f",
    ax=ax,
)
fig.tight_layout()


## Save the trained model

In [ ]:
model.save("malimg-cnn.keras")
